# Core Reranker
vLLM-Hook is an extensible framework that aims to allow selective access to model internals during the inference.
As a demonstration of that, in this notebook, we show how vLLM-Hook enables *Core Reranker* for document relevance scoring.

**Paper**: [Contrastive Retrieval Heads Improve Attention-Based Re-Ranking](https://arxiv.org/abs/2510.02219).<br />
**Authors**: Linh Tran, Yulong Li, Radu Florian, Wei Sun <br />
**"TL;DR"**: Core reranker is an attention-based reranker that leverage attention weights from selected transformer heads to produce document relevance scores.


### Installation
If running this from a fresh MIA environment, use the setup cell below from a local checkout of this repository.
The cell installs the validated dependencies and registers MIA in editable mode. Skip it if your environment is already prepared.


In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
REQ_FILE = REPO_ROOT / "requirement.txt"

os.chdir(REPO_ROOT)
repo_root_str = str(REPO_ROOT.resolve())
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

print("Repo root:", REPO_ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)])
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchcodec"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_ROOT)])
importlib.invalidate_caches()


### Importing MIA
MIA provides `MiaLLM`, a vLLM wrapper for capture and analysis.


In [ ]:
from vllm import SamplingParams, TokensPrompt
from mia import MiaLLM


### Environment & multiprocessing setup

In [ ]:
import gc
import json
import multiprocessing as mp
import os
from pathlib import Path
from typing import List

import torch
from vllm import SamplingParams

mp.set_start_method("spawn", force=True)
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Config paths in this notebook are relative to the repo root.
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())


### Helper functions that give the instruction range
As Core Reranker needs to locate the candidate passages and the user query in the prompt, below is a helper function that gives the data range with texts.<br />
Check [Core Reranker](https://arxiv.org/pdf/2510.02219) for more details.

In [ ]:
def _ids(tokenizer, text):
    """Exact ids for text, with no special tokens; spans are offsets into this sequence."""
    return tokenizer(text, add_special_tokens=False).input_ids


def apply_chat_template_and_get_ranges(tokenizer, model_name: str, query: str, documents: List[str]):
    """Return the templated prompt and the CoRe analyzer span spec."""
    retrieval_instruction = " Here are some paragraphs:\n\n"
    retrieval_instruction_late = "Please find information that are relevant to the following query in the paragraphs above.\n\nQuery: "

    content = retrieval_instruction
    doc_char_spans = []
    for i, doc in enumerate(documents):
        content += f"[document {i + 1}]"
        start_char = len(content)
        content += " " + " ".join(doc)
        doc_char_spans.append((start_char, len(content)))
        content += "\n\n"

    query_start_char = len(content)
    content += retrieval_instruction_late
    after_instruct_char = len(content)
    content += query.strip()
    query_end_char = len(content)

    messages = [{"role": "user", "content": content}]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    content_start = full_text.index(content)

    def char_to_tok(char_pos):
        prefix = full_text[:content_start + char_pos]
        return len(tokenizer(prefix, add_special_tokens=False).input_ids)

    doc_span = [(char_to_tok(s), char_to_tok(e)) for s, e in doc_char_spans]
    query_start_idx = char_to_tok(query_start_char)
    after_retrieval_instruction_late = char_to_tok(after_instruct_char)
    query_end_idx = char_to_tok(query_end_char)

    return full_text, (doc_span, query_start_idx, after_retrieval_instruction_late, query_end_idx)


### Initialize `MiaLLM`
Before we create the LLM instance, we specify the model and MIA runtime defaults.


In [ ]:
cache_dir = '../.cache'  # Specify cache dir
model = 'mistralai/Mistral-7B-Instruct-v0.3'

dtype_map = {
    'mistralai/Mistral-7B-Instruct-v0.3': torch.float16,
}

MAX_MODEL_LEN = int(os.environ.get("MIA_MAX_MODEL_LEN", "32768"))
MAX_NEW_TOKENS = int(os.environ.get("MIA_MAX_NEW_TOKENS", "1"))
GPU_MEMORY_UTILIZATION = float(os.environ.get("MIA_GPU_MEMORY_UTILIZATION", "0.70"))

quantization = None
attention_backend = None


We also need to provide a config file that specifies the important heads we want to track. <br />
For Core Reranker, this config file can be obtained from [head_detection.py](https://github.com/linhhtran/CoRe-Reranking/blob/main/experiments/head_detection.py).

In [ ]:
json_path = Path("model_configs/core_reranker/Mistral-7B-Instruct-v0.3.json")

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)


Inside MIA, `capture_qk` captures query/key tensors and `core_reranker` analyzes passage relevance.

Now, we initialize the LLM:


Before rebuilding `MiaLLM`, clear any stale CUDA allocations left by earlier notebook runs in the same session.


In [ ]:
# Clear any prior MiaLLM instance before rebuilding in the same session.
import gc

try:
    del llm
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
llm_kwargs = {
    "model": model,
    "worker_name": "capture_qk",
    "analyzer_name": "core_reranker",
    "config_file": str(json_path),
    "hook_dir": "/dev/shm/mia",
    "download_dir": cache_dir,
    "trust_remote_code": True,
    "dtype": dtype_map[model],
    "enable_hook": True,

    # MIA defaults for this notebook.
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "max_model_len": MAX_MODEL_LEN,
    "max_num_batched_tokens": MAX_MODEL_LEN,
    "max_num_seqs": 1,
    "enforce_eager": True,
    "tensor_parallel_size": 1,
    "quantization": quantization,
    "enable_prefix_caching": False,
}

if attention_backend is not None:
    llm_kwargs["attention_backend"] = attention_backend

llm = MiaLLM(**llm_kwargs)


### Test case
In the following, we show a test case with seven candidate passages and a user query.

In [ ]:
case = {
        "query": "Which came first, the invention of the telephone or the light bulb?",
        "documents": [
            [
            "Alexander Graham Bell is credited with inventing the first practical telephone.",
            " He was awarded the U.S. patent for the invention of the telephone on March 7, 1876.",
            " The first successful demonstration of the telephone took place shortly thereafter, when Bell famously called his assistant, saying, 'Mr. Watson, come here, I want to see you.'",
            " Bell’s invention revolutionized communication by allowing people to talk to each other over long distances."
            ],
            [
            "Thomas Edison is widely known for inventing the first commercially practical incandescent light bulb.",
            " Although he did not invent the concept of the light bulb itself, Edison developed a version that was safe, affordable, and long-lasting.",
            " His patent for the electric light bulb was filed in 1879, three years after Bell’s telephone patent.",
            " Edison's innovation led to widespread use of electric lighting and helped usher in the modern electrical age."
            ],
            [
            "Before Edison, several inventors worked on early versions of the light bulb.",
            " Sir Humphry Davy created the first electric arc lamp in the early 1800s, and later inventors like Joseph Swan in Britain improved upon the design.",
            " However, these early bulbs were inefficient or burned out quickly, and it was Edison who perfected the design for everyday use."
            ],
            [
            "The telephone was invented before the practical light bulb.",
            " Bell’s patent for the telephone was issued in 1876, while Edison’s patent for the light bulb was filed in 1879.",
            " Thus, the telephone came first."
            ],
            [
            "Both the telephone and the light bulb are considered groundbreaking inventions of the late 19th century.",
            " The telephone transformed communication, while the light bulb transformed how people lived and worked at night.",
            " Together, they symbolize the rapid technological progress of that era."
            ],
            [
            "Edison and Bell were contemporaries and pioneers of the Second Industrial Revolution.",
            " Their inventions marked major milestones in human history, driving the growth of telecommunications and electrical infrastructure."
            ],
            [
            "In summary, the telephone was invented in 1876 and the light bulb in 1879.",
            " Therefore, the invention of the telephone came first."
            ]
        ]
    }

Next, we apply chat template and obtain the input range using the helper function defined above.<br />
Specifically, as core reranker relies on the aggregated attentions from the user query to each passage, it needs a reference attention baseline for each passage. The authors swap the user query with `'N/A'` and treat the resulting aggregated attention as the normalizing factor for each passage.

In [ ]:
query = case["query"]
documents = case["documents"]

query_text, query_spec = apply_chat_template_and_get_ranges(llm.tokenizer, model, query, documents)
na_text, na_spec = apply_chat_template_and_get_ranges(llm.tokenizer, model, "N/A", documents)

query_ids = _ids(llm.tokenizer, query_text)
na_ids = _ids(llm.tokenizer, na_text)


Finally, we perform the model inference:

In [ ]:
sampling_params = SamplingParams(temperature=0.1, max_tokens=MAX_NEW_TOKENS)

llm.generate(
    TokensPrompt(prompt_token_ids=query_ids),
    sampling_params,
    save_to_disk=True,
    run_id="corer-doc",
)
llm.generate(
    TokensPrompt(prompt_token_ids=na_ids),
    sampling_params,
    save_to_disk=True,
    run_id="corer-na",
)


During model inference, MIA saves selected queries and keys for each run. Now, we call the analyzer to get passage relevance scores and the final passage ranking:


In [ ]:
stats = llm.analyze(
    run_ids=["corer-doc", "corer-na"],
    analyzer_spec={"query_spec": query_spec, "na_spec": na_spec},
)


Finally we can print out the results as follows:

In [ ]:
print(f"Sorted document IDs and scores by CoRe-Reranking: {stats['ranking']}: {stats['scores']}")